# 05 — App & Lookup Testing

Builds the temporary PubChem-based drug name lookup (pending DrugBank's full vocabulary export) and end-to-end tests `predict_from_names` — the same function powering both `app/api.py` and `app/streamlit_app.py`.

In [1]:
import sys
sys.path.append('..')

import requests
import json
import time

def get_smiles_from_pubchem(drug_name):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{drug_name}/property/CanonicalSMILES/TXT"
    response = requests.get(url)
    if response.status_code == 200:
        return response.text.strip()
    else:
        print(f"  Not found: {drug_name} (status {response.status_code})")
        return None

test_drugs = [
    'Warfarin', 'Aspirin', 'Sertraline', 'Phenelzine',
    'Metformin', 'Lisinopril', 'Amoxicillin', 'Atorvastatin', 'Fluconazole'
]

temp_lookup = {}
for name in test_drugs:
    smiles = get_smiles_from_pubchem(name)
    if smiles:
        temp_lookup[name.lower()] = smiles
    time.sleep(0.3)

print(f"Resolved {len(temp_lookup)} / {len(test_drugs)} drugs")


Resolved 9 / 9 drugs


In [2]:
with open('../src/drug_lookup.json', 'w') as f:
    json.dump(temp_lookup, f, indent=2)
print("Saved ../src/drug_lookup.json")


Saved ../src/drug_lookup.json


## End-to-end test

Runs the guide's own Phase 5 validation pairs through the full pipeline: name lookup → SMILES → fingerprint → prediction → translated explanation.

In [3]:
from src.predict import predict_from_names

for pair in [('Warfarin', 'Aspirin'), ('Sertraline', 'Phenelzine'),
             ('Metformin', 'Lisinopril'), ('Amoxicillin', 'Warfarin'),
             ('Atorvastatin', 'Fluconazole')]:
    try:
        label, confidence, top_features = predict_from_names(*pair)
        print(f"{pair}: {label} (confidence: {confidence:.1%})")
    except ValueError as e:
        print(f"{pair}: {e}")


('Warfarin', 'Aspirin'): Warfarin may increase the anticoagulant activities of Aspirin. (confidence: 99.3%)
('Sertraline', 'Phenelzine'): Sertraline may increase the serotonergic activities of Phenelzine. (confidence: 32.2%)
('Metformin', 'Lisinopril'): Metformin may increase the hypotensive activities of Lisinopril. (confidence: 91.0%)
('Amoxicillin', 'Warfarin'): The serum concentration of Warfarin can be decreased when it is combined with Amoxicillin. (confidence: 49.7%)
('Atorvastatin', 'Fluconazole'): The serum concentration of Fluconazole can be increased when it is combined with Atorvastatin. (confidence: 55.6%)


## Running the apps

```bash
# FastAPI backend
uvicorn app.api:app --reload

# Streamlit frontend (separate terminal)
streamlit run app/streamlit_app.py
```